# Mentoría M10 - Práctico 3
## Aprendizaje supervisado y no supervisado para reconocimiento de subambientes salares

**Proyecto:** SalarIA - clasificación inteligente de coberturas terrestres y subambientes en sistemas salares a partir de datos derivados de Sentinel-2.  
**Entrega:** 02/10/2026.  
**Modalidad:** notebook guiado tipo Coursera: gran parte del flujo está preparado y ustedes deben completar funciones puntuales marcadas con `TODO`.

### Propósito
Continuar el trabajo realizado en los Prácticos 1 y 2:

- **TP1:** exploración, visualización, bandas e índices espectrales;
- **TP2:** curación, trazabilidad, prevención de fuga de información y construcción de un dataset listo para ML;
- **TP3:** entrenamiento, evaluación e interpretación de modelos **supervisados y no supervisados**.

En este práctico se deben implementar **ambos enfoques**. Uno podrá ser presentado como enfoque principal y el otro como análisis complementario, pero los dos deben ejecutarse e interpretarse.

## Resultados de aprendizaje

Al finalizar el práctico deberían poder:

1. construir un conjunto de entrenamiento y prueba respetando grupos geográficos (`sample_id`);
2. establecer un **baseline** antes de comparar modelos más complejos;
3. entrenar al menos dos clasificadores supervisados usando `Pipeline` de scikit-learn;
4. evaluar clasificación con `F1-macro`, `balanced accuracy`, matriz de confusión y métricas por clase;
5. aplicar `KMeans` para detectar agrupamientos naturales sin utilizar la etiqueta como entrada;
6. seleccionar y justificar un número de clusters mediante `silhouette score`;
7. relacionar clusters con categorías conocidas mediante tablas cruzadas y métricas externas sin convertir las etiquetas en información de entrenamiento;
8. interpretar patrones espectrales y espaciales, limitaciones y posibles errores del modelo;
9. producir un resultado reproducible y técnicamente defendible.

## Reglas del práctico

- No utilizar `class_id`, `class_name`, `macro_class`, `sample_id`, `split`, `source`, `notes` ni campos de linaje como predictores.
- El `sample_id` se usa **solo para agrupar** observaciones y evitar fuga entre train/test.
- La etiqueta puede usarse para evaluar el modelo supervisado y para interpretar externamente el clustering, pero **nunca** como entrada de `KMeans`.
- Mantener `RANDOM_STATE = 42` salvo que se justifique otra decisión.
- No optimizar hiperparámetros usando el conjunto de test.
- Toda conclusión debe apoyarse en resultados del notebook.

> **Importante:** las celdas con `### START CODE HERE ###` y `### END CODE HERE ###` son las que deben completar. No borren las pruebas (`assert`) que siguen a esas celdas.

---
# Parte 0 - Entorno reproducible

In [ ]:
# Instalación opcional en un entorno nuevo (descomentar si hace falta)
# %pip install pandas numpy matplotlib scikit-learn

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    adjusted_rand_score,
    balanced_accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    f1_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 100)

### Buenas prácticas utilizadas

- `Path` para rutas portables;
- semillas explícitas para reproducibilidad;
- funciones pequeñas y testeables;
- `Pipeline` para evitar preprocesamiento manual inconsistente;
- `GroupShuffleSplit` para mantener `sample_id` completos en una sola partición;
- métricas macro/balanceadas porque las clases pueden tener distribuciones diferentes.

---
# Parte 1 - Carga del dataset curado del TP2

In [ ]:
# El notebook busca primero el artefacto generado en el TP2.
# Si no lo encuentra, prueba rutas habituales del repositorio.
PROJECT_ROOT = Path.cwd().parent

CANDIDATES = [
    PROJECT_ROOT / "data" / "processed" / "olaroz_samples_curated_practico2.csv",
    Path("data/processed/olaroz_samples_curated_practico2.csv"),
    PROJECT_ROOT / "tables" / "olaroz_samples_features.csv",
    Path("tables/olaroz_samples_features.csv"),
]

RAW_FALLBACK = (
    "https://raw.githubusercontent.com/pablonicolasr/mentodatos_pdis/"
    "main/tables/olaroz_samples_features.csv"
)


def cargar_dataset(candidatos: Iterable[Path]) -> tuple[pd.DataFrame, str]:
    """Carga la primera ruta disponible; como último recurso intenta GitHub Raw."""
    for ruta in candidatos:
        if ruta.exists():
            return pd.read_csv(ruta), str(ruta)

    print("No se encontró una copia local. Intentando fallback remoto...")
    return pd.read_csv(RAW_FALLBACK), RAW_FALLBACK


df, DATA_SOURCE = cargar_dataset(CANDIDATES)
print("Fuente:", DATA_SOURCE)
print("Dimensiones:", df.shape)
display(df.head())

In [ ]:
# Contrato mínimo esperado desde TP2 / dataset de features.
REQUIRED_COLUMNS = {"sample_id", "class_name", "x", "y"}
missing = REQUIRED_COLUMNS.difference(df.columns)
assert not missing, f"Faltan columnas obligatorias: {sorted(missing)}"

print("Clases:")
display(df["class_name"].value_counts().rename("n_pixeles"))
print("sample_id únicos:", df["sample_id"].nunique())

### Pregunta 1 - Trazabilidad

En una celda Markdown debajo de esta, indiquen:

- qué archivo terminó cargando el notebook;
- si corresponde al dataset curado del TP2 o a un fallback;
- qué implicancias tendría usar el fallback en lugar del producto final del TP2.

In [ ]:
# TODO ESTUDIANTE: no escribir código aquí.
# Agregar debajo una celda Markdown con la respuesta a la Pregunta 1.

---
# Parte 2 - Definición de variables y prevención de fuga de información

In [ ]:
BANDAS = ["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12"]
INDICES = ["NDVI", "NDWI", "MNDWI", "BSI", "NDBI_like"]

LEAKAGE_COLS = {
    "class_id", "class_name", "macro_class", "sample_id", "split",
    "source", "notes", "season", "confidence", "_curated_at_utc",
    "_curation_run_id", "flag_outlier_iso"
}

FEATURES = [c for c in BANDAS + INDICES if c in df.columns]

# Si el dataset curado no contiene índices, el escenario base continúa con bandas.
assert len(FEATURES) >= 6, "Se esperaban al menos seis variables espectrales disponibles."
assert not set(FEATURES).intersection(LEAKAGE_COLS)

X = df[FEATURES].copy()
y = df["class_name"].astype(str).copy()
groups = df["sample_id"].astype(str).copy()

print("Features utilizadas:", FEATURES)
print("X:", X.shape, "| y:", y.shape, "| grupos:", groups.nunique())

### Actividad 2 - Split agrupado

Completen `crear_split_agrupado`. Debe producir un train/test aproximado 80/20, **sin compartir ningún `sample_id`** entre las particiones.

No usar `train_test_split` fila a fila: varias observaciones pertenecen al mismo objeto geográfico y eso produciría una evaluación artificialmente optimista.

In [ ]:
def crear_split_agrupado(
    X: pd.DataFrame,
    y: pd.Series,
    groups: pd.Series,
    test_size: float = 0.20,
    random_state: int = RANDOM_STATE,
) -> tuple[np.ndarray, np.ndarray]:
    """Devuelve índices train/test preservando grupos completos."""

    ### START CODE HERE ###
    raise NotImplementedError("TODO: usar GroupShuffleSplit y devolver train_idx, test_idx")
    ### END CODE HERE ###

In [ ]:
# Pruebas de la Actividad 2
train_idx, test_idx = crear_split_agrupado(X, y, groups)

assert len(train_idx) > 0 and len(test_idx) > 0
assert set(train_idx).isdisjoint(set(test_idx))
assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[test_idx]))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
g_train, g_test = groups.iloc[train_idx], groups.iloc[test_idx]

print("Train:", X_train.shape)
print("Test :", X_test.shape)
print("Grupos compartidos:", len(set(g_train).intersection(set(g_test))))

### Pregunta 2 - Metodología

¿Por qué es más importante separar por `sample_id` que obtener exactamente 80% de píxeles en train y 20% en test? Respondan en una celda Markdown.

---
# Parte 3 - Baseline supervisado

In [ ]:
def evaluar_clasificador(
    nombre: str,
    modelo: BaseEstimator,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
) -> dict[str, float | str]:
    """Entrena un modelo y devuelve las métricas principales sobre test."""

    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)

    return {
        "modelo": nombre,
        "f1_macro": f1_score(y_test, pred, average="macro"),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
    }


baseline = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
resultado_baseline = evaluar_clasificador(
    "Dummy-most-frequent", baseline, X_train, y_train, X_test, y_test
)

pd.DataFrame([resultado_baseline])

### Interpretación del baseline

El baseline establece un piso explícito. Un modelo que no lo supera de manera clara no está demostrando aprendizaje útil. En datasets desbalanceados, la exactitud simple puede engañar; por eso en este práctico la métrica principal será **F1-macro**.

---
# Parte 4 - Modelos supervisados

### Actividad 3 - Regresión logística con Pipeline

Completen una función que construya:

1. imputación por mediana;
2. estandarización;
3. `LogisticRegression` multiclase con un número razonable de iteraciones.

Todo debe vivir dentro de un `Pipeline` para que el preprocesamiento se aprenda únicamente con train.

In [ ]:
def crear_pipeline_logistica(features: list[str]) -> Pipeline:
    """Pipeline de preprocesamiento + regresión logística."""

    ### START CODE HERE ###
    raise NotImplementedError("TODO: crear ColumnTransformer y LogisticRegression dentro de Pipeline")
    ### END CODE HERE ###

In [ ]:
# Pruebas de la Actividad 3
pipe_lr = crear_pipeline_logistica(FEATURES)
assert isinstance(pipe_lr, Pipeline)
assert "model" in pipe_lr.named_steps

resultado_lr = evaluar_clasificador(
    "LogisticRegression", pipe_lr, X_train, y_train, X_test, y_test
)
resultado_lr

### Actividad 4 - Random Forest

Construyan un segundo pipeline. `RandomForestClassifier` no necesita estandarización, pero sí conviene conservar una política explícita de imputación dentro del pipeline.

Usen como punto de partida:

- `n_estimators >= 200`;
- `class_weight="balanced"`;
- `random_state=RANDOM_STATE`;
- `n_jobs=-1`.

In [ ]:
def crear_pipeline_random_forest(features: list[str]) -> Pipeline:
    """Pipeline de imputación + Random Forest reproducible."""

    ### START CODE HERE ###
    raise NotImplementedError("TODO: crear preprocesamiento y RandomForestClassifier")
    ### END CODE HERE ###

In [ ]:
# Pruebas de la Actividad 4
pipe_rf = crear_pipeline_random_forest(FEATURES)
assert isinstance(pipe_rf, Pipeline)
assert "model" in pipe_rf.named_steps

resultado_rf = evaluar_clasificador(
    "RandomForest", pipe_rf, X_train, y_train, X_test, y_test
)
resultado_rf

In [ ]:
# Comparación inicial de modelos
resultados_supervisados = pd.DataFrame([
    resultado_baseline,
    resultado_lr,
    resultado_rf,
]).sort_values("f1_macro", ascending=False)

display(resultados_supervisados)

### Actividad 5 - Elegir el modelo supervisado a analizar

Seleccionen **a partir de los resultados**, no por preferencia personal, uno de los dos modelos entrenados (`pipe_lr` o `pipe_rf`) para el análisis detallado.

Completen la asignación siguiente.

In [ ]:
### START CODE HERE ###
modelo_supervisado_final = None  # reemplazar por pipe_lr o pipe_rf
### END CODE HERE ###

assert modelo_supervisado_final is not None, "Deben seleccionar un modelo final."
modelo_supervisado_final.fit(X_train, y_train)
pred_test = modelo_supervisado_final.predict(X_test)

In [ ]:
# Reporte por clase
print(classification_report(y_test, pred_test, digits=3))

# Matriz de confusión
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    pred_test,
    normalize="true",
    xticks_rotation=45,
    cmap="Blues",
    ax=ax,
)
ax.set_title("Matriz de confusión normalizada - modelo supervisado")
plt.tight_layout()
plt.show()

### Pregunta 3 - Análisis de errores

Identifiquen al menos:

- una clase bien reconocida;
- una clase problemática;
- el par de clases que más se confunde;
- una hipótesis ambiental/espectral que pueda explicar esa confusión.

No alcanza con escribir “el modelo se confunde”: relacionen el resultado con bandas, índices o similitud entre coberturas.

---
# Parte 5 - Interpretabilidad mínima del modelo supervisado

In [ ]:
# Permutation importance funciona con cualquier estimador que implemente predict/score.
# Se calcula sobre test para analizar sensibilidad de la predicción final.
perm = permutation_importance(
    modelo_supervisado_final,
    X_test,
    y_test,
    n_repeats=5,
    random_state=RANDOM_STATE,
    scoring="f1_macro",
    n_jobs=-1,
)

importance_df = (
    pd.DataFrame({
        "feature": FEATURES,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
)

display(importance_df)

fig, ax = plt.subplots(figsize=(8, 5))
plot_df = importance_df.sort_values("importance_mean")
ax.barh(plot_df["feature"], plot_df["importance_mean"], xerr=plot_df["importance_std"])
ax.set_title("Importancia por permutación - F1-macro")
ax.set_xlabel("Caída media de F1-macro al permutar")
plt.tight_layout()
plt.show()

### Pregunta 4 - Variables

¿Qué variables parecen aportar más al modelo? ¿Coincide esto con lo observado en el TP1 sobre perfiles espectrales e índices? Señalen también una limitación de interpretar importancia por permutación cuando existen variables correlacionadas.

---
# Parte 6 - Aprendizaje no supervisado

## Objetivo

Aplicar clustering sin utilizar las etiquetas durante el ajuste. La pregunta cambia:

> ¿Existen agrupamientos espectrales naturales que coincidan total o parcialmente con los subambientes definidos previamente?

Primero estandarizamos las variables porque K-Means depende de distancias.

In [ ]:
unsup_preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

X_unsup = unsup_preprocess.fit_transform(X)
print("Matriz no supervisada:", X_unsup.shape)

### Actividad 6 - Buscar un número razonable de clusters

Completen `evaluar_kmeans`. Para cada `k` debe:

1. entrenar `KMeans`;
2. obtener las etiquetas de cluster;
3. calcular `silhouette_score`;
4. devolver un DataFrame con columnas `k` y `silhouette`.

Evalúen inicialmente `k = 2, 3, 4, 5, 6`.

In [ ]:
def evaluar_kmeans(
    X_scaled: np.ndarray,
    ks: Iterable[int] = range(2, 7),
) -> pd.DataFrame:
    """Evalúa distintas cantidades de clusters con silhouette score."""

    ### START CODE HERE ###
    raise NotImplementedError("TODO: entrenar KMeans para cada k y guardar silhouette")
    ### END CODE HERE ###

In [ ]:
# Pruebas de la Actividad 6
silhouette_df = evaluar_kmeans(X_unsup)
assert {"k", "silhouette"}.issubset(silhouette_df.columns)
assert silhouette_df["k"].nunique() >= 5
assert silhouette_df["silhouette"].between(-1, 1).all()

display(silhouette_df)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(silhouette_df["k"], silhouette_df["silhouette"], marker="o")
ax.set_xlabel("Número de clusters (k)")
ax.set_ylabel("Silhouette score")
ax.set_title("Selección exploratoria de k")
ax.grid(alpha=0.3)
plt.show()

### Actividad 7 - Ajustar el K-Means final

Elijan `k` usando la evidencia anterior. **No es obligatorio que k=4 solo porque existen cuatro clases conocidas.** Si el mejor patrón interno sugiere otro valor, pueden usarlo y discutir la discrepancia.

In [ ]:
### START CODE HERE ###
K_FINAL = None  # reemplazar por un entero justificado
### END CODE HERE ###

assert isinstance(K_FINAL, int) and K_FINAL >= 2

kmeans_final = KMeans(
    n_clusters=K_FINAL,
    n_init=20,
    random_state=RANDOM_STATE,
)
clusters = kmeans_final.fit_predict(X_unsup)

print("Silhouette final:", round(silhouette_score(X_unsup, clusters), 4))

### Evaluación externa del clustering

Las siguientes métricas usan `class_name` **después** de entrenar K-Means, únicamente para estudiar correspondencia entre una partición no supervisada y las categorías existentes.

- `ARI` (Adjusted Rand Index): acuerdo entre particiones ajustado por azar.
- `NMI` (Normalized Mutual Information): información compartida entre cluster y clase.

No las usen para “entrenar” K-Means.

In [ ]:
ari = adjusted_rand_score(y, clusters)
nmi = normalized_mutual_info_score(y, clusters)

print(f"ARI: {ari:.4f}")
print(f"NMI: {nmi:.4f}")

cluster_vs_class = pd.crosstab(
    pd.Series(clusters, name="cluster"),
    y.reset_index(drop=True).rename("class_name"),
    normalize="index",
).round(3)

display(cluster_vs_class)

### Pregunta 5 - Correspondencia ambiental

Analicen la tabla `cluster_vs_class` y respondan:

- ¿hay clusters dominados por una clase?
- ¿hay clases distribuidas en varios clusters?
- ¿el clustering complementa, reproduce o cuestiona las categorías ambientales existentes?
- ¿podría un cluster estar capturando gradientes internos de humedad, salinidad o sustrato en lugar de una clase semántica completa?

---
# Parte 7 - Visualización en espacio reducido y espacio geográfico

In [ ]:
# PCA se usa aquí solo para visualización 2D.
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_unsup)

viz = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "class_name": y.values,
    "cluster": clusters,
    "x": df["x"].values,
    "y": df["y"].values,
})

print("Varianza explicada PCA 2D:", pca.explained_variance_ratio_.sum().round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for cluster_id in sorted(viz["cluster"].unique()):
    sub = viz[viz["cluster"] == cluster_id]
    ax.scatter(sub["PC1"], sub["PC2"], s=7, alpha=0.35, label=f"cluster {cluster_id}")
ax.set_title("Clusters K-Means proyectados con PCA")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(viz["x"], viz["y"], c=viz["cluster"], s=6, alpha=0.55, cmap="tab10")
ax.set_title("Distribución espacial de clusters")
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.colorbar(scatter, ax=ax, label="cluster")
plt.tight_layout()
plt.show()

### Pregunta 6 - Estructura espacial

¿Los clusters forman zonas geográficas coherentes o aparecen completamente mezclados? ¿Una estructura espacial fuerte podría representar señal ambiental real, dependencia espacial o ambas cosas? Relacionen esta discusión con la decisión de separar train/test por `sample_id`.

---
# Parte 8 - Comparación supervisado vs no supervisado

In [ ]:
resumen = pd.DataFrame([
    {
        "enfoque": "Supervisado - modelo final",
        "f1_macro": f1_score(y_test, pred_test, average="macro"),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred_test),
        "silhouette": np.nan,
        "ARI_vs_clase": np.nan,
        "NMI_vs_clase": np.nan,
    },
    {
        "enfoque": f"No supervisado - KMeans k={K_FINAL}",
        "f1_macro": np.nan,
        "balanced_accuracy": np.nan,
        "silhouette": silhouette_score(X_unsup, clusters),
        "ARI_vs_clase": ari,
        "NMI_vs_clase": nmi,
    },
])

display(resumen)

## Informe final dentro del notebook

Agregar una sección Markdown titulada **Conclusiones del TP3** con una extensión aproximada de 400 a 700 palabras. Debe responder, como mínimo:

1. ¿Qué baseline se utilizó y por qué?
2. ¿Qué modelo supervisado obtuvo mejores resultados y con qué métricas?
3. ¿Qué clases fueron más fáciles/difíciles de reconocer?
4. ¿Qué `k` se eligió y qué evidencia lo respalda?
5. ¿Qué relación encontraron entre clusters y categorías conocidas?
6. ¿Qué variables parecen relevantes?
7. ¿Cuáles son las principales limitaciones metodológicas?
8. ¿Cuál sería el siguiente experimento para mejorar la clasificación inteligente de subambientes salares?

---
# Extensiones opcionales (+ valor técnico)

Se puede agregar **una o más** de las siguientes extensiones, siempre justificadas:

- validación cruzada agrupada (`GroupKFold` o `StratifiedGroupKFold` si la versión de sklearn lo permite);
- búsqueda de hiperparámetros usando exclusivamente train/CV;
- comparación `KMeans` vs clustering jerárquico o DBSCAN/HDBSCAN;
- incorporación controlada de índices espectrales adicionales;
- comparación de escenarios “solo bandas” vs “bandas + índices”;
- mapa de errores del clasificador;
- análisis de centroides espectrales de los clusters;
- evaluación de estabilidad del clustering con diferentes semillas;
- separación geográfica más estricta para estudiar generalización espacial;
- exportación de predicciones/clusters a CSV para uso posterior en QGIS.

> Una extensión suma valor si responde una pregunta. Agregar modelos sin interpretación no constituye por sí mismo una mejora.

---
# Entregables

1. `M10_Practico3_Aprendizaje_ML_GRUPO_N.ipynb`, ejecutado de punta a punta y con respuestas.
2. `predicciones_tp3_grupo_N.csv`, con al menos `sample_id`, clase real, predicción supervisada y cluster.
3. Un breve informe técnico en PDF (máx. 4 páginas) o la sección final del notebook exportada a PDF, según indique la cátedra.

La celda siguiente deja preparada la exportación mínima de predicciones.

In [ ]:
# Exportación mínima. Se ejecuta después de completar todo el TP.
SALIDA = Path("predicciones_tp3_grupo_N.csv")

resultado_test = df.iloc[test_idx][["sample_id", "x", "y", "class_name"]].copy()
resultado_test["pred_supervisada"] = pred_test
resultado_test["cluster"] = clusters[test_idx]
resultado_test.to_csv(SALIDA, index=False)

print("Archivo generado:", SALIDA.resolve())
display(resultado_test.head())

# Checklist antes de entregar

- [ ] Todas las celdas ejecutan en orden sin errores.
- [ ] No quedan `NotImplementedError` ni `None` en actividades obligatorias.
- [ ] No existe fuga de `sample_id` entre train y test.
- [ ] Se reporta y discute el baseline.
- [ ] Se comparan al menos dos clasificadores.
- [ ] Se analiza matriz de confusión y métricas por clase.
- [ ] Se aplica K-Means y se justifica `k`.
- [ ] Se interpreta la relación cluster/clase sin usar la etiqueta como feature.
- [ ] Se incluye visualización PCA y espacial.
- [ ] Se presentan conclusiones y limitaciones.
- [ ] El CSV de predicciones fue generado.